# Notebook 03 — Model Training

## What does this notebook do?

We train a ResNet-18 neural network to classify breast ultrasound images
into 3 classes: benign, malignant, normal.

## Training happens in 2 stages:

STAGE 1 — Frozen backbone (10-15 epochs)
```
  We keep the pretrained ResNet-18 weights FROZEN
  Only the new classification head is trained
  Why? The new head has random weights — training everything at once
  would destroy the useful pretrained features
```

STAGE 2 — Fine-tuning (up to 25 epochs)
```
  We UNFREEZE all layers
  We use a very small learning rate (1e-4)
  Why? We want to make small adjustments to the pretrained weights
  to better suit ultrasound images specifically
```

## After every epoch we:
- Check performance on the VALIDATION set
- Save the best model checkpoint
- Stop early if the model stops improving

---
##  Import Libraries

In [ ]:
import os
import torch                            # main deep learning library
import torch.nn as nn                   # neural network layers
import pandas as pd                     # to save training log as CSV
import matplotlib.pyplot as plt         # to plot training curves
from pathlib import Path
from tqdm import tqdm                   # progress bar during training
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler

# Fix random seeds so results are reproducible
torch.manual_seed(42)

print('Libraries imported successfully!')

# Check if a GPU is available (training is much faster on GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('  cuda = GPU is available (fast)')
print('  cpu  = no GPU, will use CPU (slower but works fine)')

---
## Define Image Transforms

A transform is a set of operations applied to each image before
it is fed into the model.

TRAIN transform includes AUGMENTATION:
  - Random flip, rotation, brightness changes
  - This artificially creates variation in the training data
  - Helps the model generalise better to new images

VAL and TEST transforms have NO augmentation:
  - Only resize and normalise
  - We want honest evaluation — no artificial changes

In [ ]:
# ImageNet normalisation values
# We use these because our model was pretrained on ImageNet
# The model expects inputs in this exact range
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


# ── TRAIN transform — includes augmentation ──────────────────
train_transform = transforms.Compose([

    # Ultrasound images are grayscale (1 channel)
    # ResNet-18 needs 3 channels — we duplicate the single channel 3 times
    transforms.Grayscale(num_output_channels=3),

    # Resize to 224x224 — required by ResNet-18
    transforms.Resize((224, 224)),

    # Flip image left-right with 50% probability
    # Lesions can appear on either side so this is clinically valid
    transforms.RandomHorizontalFlip(p=0.5),

    # Rotate image by up to 15 degrees
    # Mimics slight variation in probe angle during scanning
    transforms.RandomRotation(degrees=15),

    # Slightly change brightness and contrast
    # Mimics variation between different ultrasound machines
    transforms.ColorJitter(brightness=0.2, contrast=0.2),

    # Convert PIL image to PyTorch tensor
    transforms.ToTensor(),

    # Normalise pixel values using ImageNet statistics
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])


# ── VAL and TEST transform — NO augmentation ─────────────────
val_test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])


print('Transforms defined!')
print()
print('Train transform   : resize + flip + rotation + brightness + normalise')
print('Val/Test transform: resize + normalise only')

---
##  Load the Datasets

ImageFolder automatically reads images from the folder structure
we created in notebook 02 and assigns class labels.

We also create a WeightedRandomSampler for the training set.
This makes sure each mini-batch contains a balanced mix of classes
even though benign images are far more common in the dataset.

In [ ]:
# ── Load images from the split folders ───────────────────────
train_dataset = datasets.ImageFolder('data/processed/train', transform=train_transform)
val_dataset   = datasets.ImageFolder('data/processed/val',   transform=val_test_transform)
test_dataset  = datasets.ImageFolder('data/processed/test',  transform=val_test_transform)

print(f'Classes detected : {train_dataset.classes}')
print(f'  benign=0   malignant=1   normal=2')
print()
print(f'Train size : {len(train_dataset)} images')
print(f'Val   size : {len(val_dataset)} images')
print(f'Test  size : {len(test_dataset)} images')
print()


# ── WeightedRandomSampler for class imbalance ─────────────────
#
# Problem: benign=56%, malignant=27%, normal=17%
# Without correction, the model will mostly see benign images
# and become biased toward predicting benign.
#
# Solution: give rare classes a higher chance of being picked
# in each mini-batch. A normal image is picked more often
# than a benign image to balance the training.

# Count how many images exist per class in the training set
class_counts = [0, 0, 0]   # [benign_count, malignant_count, normal_count]
for _, label in train_dataset.samples:
    class_counts[label] += 1

print(f'Training class counts : {train_dataset.classes}')
print(f'                        {class_counts}')

# Calculate weight for each class
# A class with fewer images gets a HIGHER weight
# Example: if normal has 93 images -> weight = 1/93 = 0.011
#          if benign has 306 images -> weight = 1/306 = 0.003
#          so normal images are picked ~3x more often
class_weights = [1.0 / c for c in class_counts]

# Assign a weight to every single image in the training set
sample_weights = [class_weights[label] for _, label in train_dataset.samples]

# Create the sampler
sampler = WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(sample_weights),
    replacement = True   # same image can appear more than once per epoch
)

print()
print('WeightedRandomSampler created successfully!')


# ── Create DataLoaders ────────────────────────────────────────
#
# DataLoader feeds images to the model in batches of 32
# batch_size=32 means the model sees 32 images at once before
# updating its weights

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f'Train loader : {len(train_loader)} batches of 32')
print(f'Val   loader : {len(val_loader)} batches of 32')
print(f'Test  loader : {len(test_loader)} batches of 32')

---
## Build the Model

We use ResNet-18 pretrained on ImageNet.

What is ResNet-18?
- A convolutional neural network with 18 layers
- Already trained on 1.2 million images (ImageNet)
- It has learned to detect edges, shapes, textures
- We replace the final layer (which predicted 1000 ImageNet classes)
  with a new layer that predicts our 3 classes

What is freeze_backbone?
- True  = only train the new head (Stage 1)
- False = train everything (Stage 2)

In [ ]:
def build_model(freeze_backbone=True):
    """
    Load pretrained ResNet-18 and replace the final layer
    with a new 3-class classification head.

    freeze_backbone=True  -> Stage 1: only train the new head
    freeze_backbone=False -> Stage 2: train all layers
    """

    # Load ResNet-18 with pretrained ImageNet weights
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        # Freeze all layers — their weights will NOT change during training
        for param in model.parameters():
            param.requires_grad = False
        print('Backbone FROZEN — only the new head will be trained')
    else:
        print('Backbone UNFROZEN — all layers will be trained')

    # ResNet-18 final layer: Linear(512 -> 1000)
    # We replace it with:    Linear(512 -> 3)
    # 512 = number of features coming from the backbone
    # 3   = our number of classes (benign, malignant, normal)
    in_features = model.fc.in_features   # this is 512 for ResNet-18

    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),               # randomly zero out 40% of neurons
                                         # during training to reduce overfitting
        nn.Linear(in_features, 3)        # new 3-class output layer
    )

    return model


# Build the model and move it to GPU (if available)
model = build_model(freeze_backbone=True).to(device)

# Count how many parameters will be trained
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print()
print(f'Trainable parameters : {trainable:,}')
print(f'Total parameters     : {total:,}')

---
## Define Loss Function and Optimiser

LOSS FUNCTION:
  CrossEntropyLoss measures how wrong the model's predictions are.
  We add class weights so that mistakes on rare classes
  (malignant, normal) cost more than mistakes on common classes.

OPTIMISER:
  Adam is the algorithm that updates the model weights
  after each batch to reduce the loss.

SCHEDULER:
  Reduces the learning rate by half if the validation loss
  has not improved for 3 epochs in a row.

In [ ]:
# ── Loss function with class weights ─────────────────────────
# Higher weight = a mistake on this class costs more
# This encourages the model to pay more attention to rare classes
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

print('Loss function : CrossEntropyLoss with class weights')
print(f'  benign weight    : {class_weights[0]:.4f}  (most common -> lowest weight)')
print(f'  malignant weight : {class_weights[1]:.4f}')
print(f'  normal weight    : {class_weights[2]:.4f}  (least common -> highest weight)')
print()


# ── Optimiser — Stage 1 learning rate ────────────────────────
# filter(requires_grad) means we only pass the trainable parameters
# In Stage 1 that is only the new head
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3    # learning rate: how big each weight update step is
)

print('Optimiser : Adam   lr=0.001   (Stage 1)')


# ── Learning Rate Scheduler ───────────────────────────────────
# If val_loss does not improve for 3 epochs, cut lr in half
# This helps the model make finer adjustments as training progresses
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode      = 'min',   # we want val_loss to go DOWN
    factor    = 0.5,     # multiply lr by 0.5 when triggered
    patience  = 3        # wait 3 epochs before reducing
)

print('Scheduler : ReduceLROnPlateau  factor=0.5  patience=3')

---
## Early Stopping

Early stopping watches the validation loss after every epoch.
If the loss has not improved for PATIENCE epochs in a row,
training stops automatically.

It also saves the BEST model checkpoint automatically.
Best = the epoch where validation loss was lowest.

In [ ]:
class EarlyStopping:
    """
    Stops training if validation loss does not improve.
    Saves the best model automatically.
    """

    def __init__(self, patience=7, save_path='outputs/models/best_model.pth'):
        """
        patience  : how many epochs to wait before stopping
        save_path : where to save the best model weights
        """
        self.patience     = patience
        self.save_path    = save_path
        self.best_loss    = float('inf')   # start with a very large number
        self.counter      = 0              # counts epochs without improvement
        self.should_stop  = False          # flag to signal training to stop

    def check(self, val_loss, model):
        """
        Call this after every epoch.
        val_loss : the validation loss from this epoch
        model    : the current model (to save if this is the best epoch)
        """

        if val_loss < self.best_loss:
            # Loss improved — save this model and reset counter
            self.best_loss = val_loss
            self.counter   = 0
            os.makedirs(os.path.dirname(self.save_path), exist_ok=True)
            torch.save(model.state_dict(), self.save_path)
            print(f'    Val loss improved  ->  model saved to {self.save_path}')
        else:
            # No improvement — increment counter
            self.counter += 1
            print(f'    No improvement  ({self.counter}/{self.patience})')
            if self.counter >= self.patience:
                self.should_stop = True
                print('    Early stopping triggered!')


print('EarlyStopping class defined!')

---
## Training and Validation Functions

train_one_epoch:
  Loops through all training batches.
  For each batch: forward pass -> calculate loss -> backprop -> update weights.
  Returns average loss and accuracy for the epoch.

validate:
  Loops through all validation batches.
  No weight updates — we only measure performance.
  Returns average loss and accuracy.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """
    Run one full pass through the training data.
    Returns: (average_loss, accuracy)
    """

    model.train()   # set model to training mode (enables dropout etc.)

    total_loss = 0.0
    correct    = 0
    total      = 0

    # tqdm wraps the loader to show a progress bar
    for images, labels in tqdm(loader, desc='  Training', leave=False):

        # Move data to GPU if available
        images = images.to(device)
        labels = labels.to(device)

        # Step 1: Forward pass — feed images through the model
        predictions = model(images)

        # Step 2: Calculate how wrong the predictions are
        loss = criterion(predictions, labels)

        # Step 3: Zero out old gradients from previous batch
        optimizer.zero_grad()

        # Step 4: Backpropagation — calculate gradients
        loss.backward()

        # Step 5: Update model weights using gradients
        optimizer.step()

        # Track total loss and correct predictions
        total_loss += loss.item() * images.size(0)
        correct    += (predictions.argmax(1) == labels).sum().item()
        total      += images.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


def validate(model, loader, criterion, device):
    """
    Run one full pass through the validation data.
    No weight updates — only measure performance.
    Returns: (average_loss, accuracy)
    """

    model.eval()   # set model to evaluation mode (disables dropout)

    total_loss = 0.0
    correct    = 0
    total      = 0

    # torch.no_grad() tells PyTorch not to track gradients
    # This saves memory and speeds up validation
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='  Validating', leave=False):
            images = images.to(device)
            labels = labels.to(device)

            predictions = model(images)
            loss        = criterion(predictions, labels)

            total_loss += loss.item() * images.size(0)
            correct    += (predictions.argmax(1) == labels).sum().item()
            total      += images.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


print('Training and validation functions defined!')

---
##  STAGE 1: Train the Classification Head Only

Backbone is FROZEN — only the new 3-class head is trained.
Learning rate: 0.001
Max epochs: 15
Early stopping patience: 7

Watch the val_loss column — it should decrease over epochs.
When it stops decreasing, early stopping will trigger.

In [ ]:
print('=' * 65)
print('  STAGE 1: Training classification head (backbone frozen)')
print('=' * 65)

early_stopping = EarlyStopping(
    patience  = 7,
    save_path = 'outputs/models/best_model_stage1.pth'
)

# Store results for plotting later
log = []

for epoch in range(1, 16):   # epochs 1 to 15

    # Run one training epoch
    train_loss, train_acc = train_one_epoch(model, train_loader,
                                            optimizer, criterion, device)

    # Run validation
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    # Update learning rate if needed
    scheduler.step(val_loss)

    # Print this epoch's results
    print(f'Epoch {epoch:2d}/15  '
          f'Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.3f}  |  '
          f'Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.3f}')

    # Check early stopping
    early_stopping.check(val_loss, model)

    # Save to log
    log.append({
        'stage'      : 1,
        'epoch'      : epoch,
        'train_loss' : round(train_loss, 4),
        'train_acc'  : round(train_acc,  4),
        'val_loss'   : round(val_loss,   4),
        'val_acc'    : round(val_acc,    4)
    })

    if early_stopping.should_stop:
        print(f'  Stopped early at epoch {epoch}')
        break

print()
print('Stage 1 complete!')

---
##  STAGE 2: Fine-Tune the Full Network

Now we UNFREEZE all layers and train everything.
We use a much smaller learning rate (0.0001) so we only
make small adjustments to the pretrained weights.

In [ ]:
print('=' * 65)
print('  STAGE 2: Fine-tuning full network (all layers unfrozen)')
print('=' * 65)

# Unfreeze ALL layers
for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'All layers unfrozen — trainable parameters: {trainable:,}')
print()

# New optimiser with smaller learning rate for fine-tuning
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# New scheduler and early stopping for Stage 2
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

early_stopping = EarlyStopping(
    patience  = 7,
    save_path = 'outputs/models/best_model_stage2.pth'
)

for epoch in range(1, 26):   # up to 25 more epochs

    train_loss, train_acc = train_one_epoch(model, train_loader,
                                            optimizer, criterion, device)
    val_loss, val_acc     = validate(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    print(f'Epoch {epoch:2d}/25  '
          f'Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.3f}  |  '
          f'Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.3f}')

    early_stopping.check(val_loss, model)

    log.append({
        'stage'      : 2,
        'epoch'      : epoch,
        'train_loss' : round(train_loss, 4),
        'train_acc'  : round(train_acc,  4),
        'val_loss'   : round(val_loss,   4),
        'val_acc'    : round(val_acc,    4)
    })

    if early_stopping.should_stop:
        print(f'  Stopped early at epoch {epoch}')
        break

print()
print('Stage 2 complete!')
print(f'Best model saved -> outputs/models/best_model_stage2.pth')

---
##  Save Training Log and Plot Training Curves

We save all epoch results to a CSV file.
Then we plot the training curves — this becomes Figure 3 in your report.

What to look for in the curves:
- Train loss and val loss should both go DOWN over time
- If val loss starts going UP while train loss keeps going DOWN
  that means OVERFITTING (model memorising training data)

In [ ]:
os.makedirs('outputs/logs', exist_ok=True)

# Save training log to CSV
df = pd.DataFrame(log)
df.to_csv('outputs/logs/training_log.csv', index=False)
print('Training log saved -> outputs/logs/training_log.csv')
print()
print(df.tail(5).to_string(index=False))   # show last 5 rows
print()

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Figure 3: Training and Validation Curves', fontsize=13, fontweight='bold')

epoch_nums = list(range(1, len(log) + 1))

# ── LEFT: Loss curves ────────────────────────────────────────
axes[0].plot(epoch_nums, df['train_loss'], label='Train Loss', color='steelblue')
axes[0].plot(epoch_nums, df['val_loss'],   label='Val Loss',   color='tomato')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── RIGHT: Accuracy curves ───────────────────────────────────
axes[1].plot(epoch_nums, df['train_acc'], label='Train Accuracy', color='steelblue')
axes[1].plot(epoch_nums, df['val_acc'],   label='Val Accuracy',   color='tomato')
axes[1].set_title('Accuracy over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('outputs/figures', exist_ok=True)
plt.savefig('outputs/figures/fig3_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print('Figure 3 saved -> outputs/figures/fig3_training_curves.png')

---
## Load Best Model and Quick Validation Check

Load the best saved checkpoint and run one final
validation check to confirm the best model is loaded correctly.

In [ ]:
# Load the best model weights saved during Stage 2
model.load_state_dict(torch.load('outputs/models/best_model_stage2.pth',
                                  map_location=device))
print('Best model loaded from outputs/models/best_model_stage2.pth')

# Run one final validation pass to confirm
val_loss, val_acc = validate(model, val_loader, criterion, device)
print(f'Final val loss : {val_loss:.4f}')
print(f'Final val acc  : {val_acc:.4f}')
print()
print('Model is ready for evaluation!')
print()
print('NEXT STEP: Run 04_Evaluation.ipynb')
print('That notebook runs the model on the TEST set and reports all metrics.')